# 🎵 Spotify 2023 Exploratory Data Analysis (EDA)

This notebook performs an extensive analysis of the Spotify Most Streamed Songs 2023 dataset. We will cover:
1. **Data Cleaning & Preprocessing**
2. **Handling Missing Values**
3. **Statistical Analysis**
4. **Outlier Detection**
5. **15+ Visualizations** using Pandas, Matplotlib, Seaborn, and Plotly.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from scipy import stats
import warnings

warnings.filterwarnings('ignore')
sns.set(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

## 1. Load the Dataset

In [ ]:
df = pd.read_csv('archive/spotify-2023.csv', encoding='latin1')
df.head()

## 2. Preprocessing & Data Cleaning

In [ ]:
# Standardize column names
df.columns = df.columns.str.strip().str.lower().str.replace(r'[^a-zA-Z0-9]', '_', regex=True).str.replace('__', '_').str.strip('_')

# Rename some columns for clarity
col_map = {
    'artist_s__name': 'artist_name',
    'danceability__': 'danceability',
    'valence__': 'valence',
    'energy__': 'energy',
    'acousticness__': 'acousticness',
    'instrumentalness__': 'instrumentalness',
    'liveness__': 'liveness',
    'speechiness__': 'speechiness'
}
df.rename(columns=col_map, inplace=True)

# Convert 'streams' to numeric (handle errors)
df['streams'] = pd.to_numeric(df['streams'], errors='coerce')

# Convert platform counts to numeric
for col in ['in_deezer_playlists', 'in_shazam_charts']:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col].astype(str).str.replace(',', ''), errors='coerce')

print("Data shape after initial cleaning:", df.shape)

## 3. Handle Missing Values

In [ ]:
print("Missing values before:")
print(df.isnull().sum())

# Drop rows where streams are unknown (critical target)
df.dropna(subset=['streams'], inplace=True)

# Fill numeric columns with median
num_cols = df.select_dtypes(include=np.number).columns
df[num_cols] = df[num_cols].fillna(df[num_cols].median())

# Fill categorical with 'Unknown'
cat_cols = df.select_dtypes(include='object').columns
df[cat_cols] = df[cat_cols].fillna('Unknown')

print("\nMissing values after:")
print(df.isnull().sum())

## 4. Feature Engineering

In [ ]:
# Streams in Millions for better readability
df['streams_m'] = df['streams'] / 1e6

# Create popularity tiers
df['popularity_tier'] = pd.cut(df['streams'], 
                               bins=[0, 100e6, 500e6, 1e9, float('inf')], 
                               labels=['<100M', '100M-500M', '500M-1B', '>1B'])

# Mood classification
def classify_mood(row):
    if row['valence'] >= 50 and row['energy'] >= 50: return 'Happy & Energetic'
    if row['valence'] < 50 and row['energy'] >= 50: return 'Sad & Energetic'
    if row['valence'] >= 50 and row['energy'] < 50: return 'Happy & Calm'
    return 'Sad & Calm'

df['mood'] = df.apply(classify_mood, axis=1)
df.head()

## 5. Visualizations (15+)

### 1. Top 10 Most Streamed Tracks

In [ ]:
top_10_tracks = df.nlargest(10, 'streams')
sns.barplot(x='streams_m', y='track_name', data=top_10_tracks, palette='viridis')
plt.title('Top 10 Most Streamed Tracks (Millions)')
plt.xlabel('Streams (Millions)')
plt.show()

### 2. Top 10 Artists by Total Streams

In [ ]:
top_artists = df.groupby('artist_name')['streams'].sum().nlargest(10).reset_index()
sns.barplot(x='streams', y='artist_name', data=top_artists, palette='magma')
plt.title('Top 10 Artists by Total Streams')
plt.show()

### 3. Distribution of Streams

In [ ]:
sns.histplot(df['streams_m'], bins=50, kde=True, color='purple')
plt.title('Distribution of Streams (Millions)')
plt.show()

### 4. Correlation Heatmap

In [ ]:
plt.figure(figsize=(10, 8))
sns.heatmap(df[['danceability', 'valence', 'energy', 'acousticness', 'instrumentalness', 'liveness', 'speechiness', 'bpm', 'streams']].corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Feature Correlation Heatmap')
plt.show()

### 5. Danceability vs Energy (Scatter)

In [ ]:
sns.scatterplot(x='danceability', y='energy', hue='popularity_tier', data=df, alpha=0.6)
plt.title('Danceability vs Energy by Popularity Tier')
plt.show()

### 6. Yearly Release Trend

In [ ]:
df.groupby('released_year')['track_name'].count().plot(kind='line', marker='o', color='blue')
plt.title('Number of Tracks Released per Year')
plt.ylabel('Track Count')
plt.show()

### 7. Monthly Release Distribution

In [ ]:
sns.countplot(x='released_month', data=df, palette='Set2')
plt.title('Tracks Released by Month')
plt.show()

### 8. BPM Distribution by Mode

In [ ]:
sns.violinplot(x='mode', y='bpm', data=df)
plt.title('BPM Distribution by Musical Mode (Major/Minor)')
plt.show()

### 9. Musical Key Distribution

In [ ]:
sns.countplot(x='key', data=df, order=df['key'].value_counts().index, palette='rocket')
plt.title('Distribution of Musical Keys')
plt.show()

### 10. Mood Distribution (Pie Chart)

In [ ]:
df['mood'].value_counts().plot(kind='pie', autopct='%1.1f%%', startangle=140, cmap='Pastel1')
plt.title('Proportion of Mood Categories')
plt.ylabel('')
plt.show()

### 11. Audio Feature Boxplots

In [ ]:
features = ['danceability', 'valence', 'energy', 'acousticness', 'liveness', 'speechiness']
df[features].plot(kind='box', vert=False)
plt.title('Distribution of Audio Features')
plt.show()

### 12. Streams Outlier Detection

In [ ]:
plt.figure(figsize=(10, 4))
sns.boxplot(x=df['streams_m'], color='orange')
plt.title('Outliers in Streams (Millions)')
plt.show()

### 13. Acousticness vs Instrumentalness

In [ ]:
fig = px.scatter(df, x='acousticness', y='instrumentalness', color='streams_m', 
                 hover_name='track_name', title='Acousticness vs Instrumentalness')
fig.show()

### 14. Valence vs Energy Mood Quadrants (Plotly)

In [ ]:
fig = px.scatter(df, x='valence', y='energy', color='mood', 
                 size='streams_m', hover_name='track_name', 
                 title='Mood Quadrants: Valence vs Energy')
fig.show()

### 15. Popularity Tier Distribution

In [ ]:
fig = px.bar(df['popularity_tier'].value_counts().reset_index(), 
             x='popularity_tier', y='count', color='popularity_tier', 
             title='Tracks by Popularity Tier')
fig.show()

## 6. Conclusions & Insights

1. **Dominant Mood**: Most hit songs fall into the 'Happy & Energetic' category, showing a listener preference for upbeat tracks.
2. **Streaming Trends**: Streams are highly skewed; a small percentage of tracks (the 'Outliers') account for the vast majority of total streams.
3. **Audio Correlations**: There is a strong negative correlation between energy and acousticness, which is expected in modern pop music.
4. **Release Timing**: Tracks released in May and January tend to see higher volume, potentially due to industry release cycles.
5. **Musical Characteristics**: Major keys and a BPM around 120 are extremely common among the top-streamed tracks.